# Урок 02 - Изучение Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** — это унифицированная платформа для создания ИИ-агентов. Она предоставляет чистую, компонуемую архитектуру с четырьмя основными строительными блоками:

- **Client** – подключается к конечной точке модели ИИ и обрабатывает коммуникацию
- **Agent** – оборачивает клиент с инструкциями и определениями инструментов
- **Tools** – расширяют возможности агента с помощью пользовательских функций, которые модель может вызывать
- **Session** – сохраняет историю разговора для многоэтапного взаимодействия

В этом уроке мы создадим **агента для бронирования путешествий**, который проверяет доступность направления, используя эти концепции.


## Настройка


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Понимание архитектуры Agent Framework

Microsoft Agent Framework использует многоуровневую архитектуру:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` подключается к развертыванию Azure OpenAI. Он обрабатывает аутентификацию, форматирование запросов и парсинг ответов.
2. **Agent** – Создается из клиента с помощью `provider.create_agent()`. Агент объединяет доступ к модели с инструкциями (системным приглашением) и инструментами.
3. **Tools** – Функции Python с декоратором `@tool`, которые агент может вызывать для выполнения действий или получения данных.
4. **Session** – Объект `AgentSession` (созданный через `agent.create_session()`), который хранит историю разговора, обеспечивая многоходовой диалог, где агент помнит предыдущий контекст.

Давайте создадим каждый уровень поэтапно.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Добавление инструментов с декоратором @tool

Инструменты позволяют агентам выполнять действия, выходящие за рамки генерации текста. Декоратор `@tool` преобразует обычную функцию Python в нечто, что агент может вызывать.

Основные моменты:
- Используйте `Annotated[type, "описание"]`, чтобы модель понимала каждый параметр.
- Докстринг становится описанием инструмента, которое видит модель.
- `approval_mode="never_require"` означает, что инструмент запускается автоматически без подтверждения пользователя.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Создание агента с инструментами

Теперь мы объединяем клиента, инструкции и инструменты в агента. `instructions` выступают в роли системного запроса — они определяют личность и поведение агента.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Многошаговые разговоры с сессиями

`AgentSession` (создаётся с помощью `agent.create_session()`) отслеживает все сообщения в разговоре. Передавая одну и ту же сессию в каждый вызов `agent.run()`, агент получает доступ ко всей истории разговора и может ссылаться на предыдущие сообщения.

Мы передаём `tools=[check_destination_availability]`, чтобы агент мог вызывать наш проверщик доступности на каждом шаге.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Резюме

В этом уроке вы изучили четыре столпа Microsoft Agent Framework:

| Концепция | Что вы узнали |
|---------|------------------|
| **Клиент** | `AzureAIProjectAgentProvider` подключается к Azure OpenAI с аутентификацией на основе учетных данных |
| **Агент** | `provider.create_agent()` объединяет подключение к модели с инструкциями и именем |
| **Инструменты** | Декоратор `@tool` открывает функции Python для вызова агентом |
| **Сессия** | `agent.create_session()` поддерживает историю разговора на нескольких ходах |

Эти строительные блоки объединяются для создания агентов, которые могут вести естественные разговоры, вызывать внешние функции и сохранять контекст — основу для более продвинутых агентских паттернов в последующих уроках.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Данный документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на то, что мы стремимся к точности, просим учитывать, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильное толкование, возникающие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
